# Assignment 5 - Vector Store


## FP1 Missing Content
The first fail case is when asking a question that cannot be answered from the available documents. In the happy case the RAG system will respond with something like “Sorry, I don’t know". However, for questions that
are related to the content but don’t have answers the system
could be fooled into giving a response.

## FP2 Missed the Top Ranked Documents
The answer to the question is in the document but did not rank highly enough to be returned to the user. In theory, all documents are ranked and used in the next steps. However, in practice the top K documents are returned where K is a value selected based on performance.

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
%%capture
!pip install -q -U langchain langchain-community langchain-huggingface langchain-qdrant
!pip install -q -U qdrant-client sentence-transformers arxiv pymupdf
!pip install -q -U transformers accelerate bitsandbytes

!pip install -q -U xmltodict

!pip install -q -U cohere

!pip install -q -U langchain-cohere

!pip install -q -U wikipedia


In [3]:
import os
import numpy as np
import time
import locale
from google.colab import userdata

# IMPORTANT: Add your Hugging Face token to Colab's Secrets (the key icon on the left panel)
# and name it 'HF_TOKEN', or replace the line below with os.environ["HF_TOKEN"] = "your_token"
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
COHERE_API_KEY = userdata.get('COHERE_API_KEY')


import os
os.environ["USER_AGENT"] = "RAG_Assignment/v0.1 (davidschaaf@berkeley.edu)"

!mkdir -p /content/qdrant_storage

import langchain
from langchain_community.document_loaders import ArxivLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_qdrant import QdrantVectorStore

from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.utils.function_calling import convert_to_openai_tool


from langchain_community.document_loaders import ArxivLoader
from langchain_community.document_loaders import PyMuPDFLoader

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig


from langchain_cohere import ChatCohere

import torch.nn.functional as F
from torch import Tensor
from transformers import AutoModel

import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import WikipediaLoader


/tmp/ipykernel_1119/2247511634.py:19: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import ArxivLoader


## 2. Building the Components of our RAG System

Let us introduce and test the base components of our RAG system. We will largely use the Hugging Face and LangChain libraries.



### 2.1 The Embedding Model

We will need to represent text (pieces) as vectors. For this, we will use the [sentence_transformer](https://sbert.net/docs/sentence_transformer/pretrained_models.html) architecture.

**NOTE:** The embedding models you can use are: 'all-mpnet-base-v2', 'all-MiniLM-L12-v2', 'multi-qa-mpnet-base-dot-v1', 'all-distilroberta-v1', and 'multi-qa-distilbert-cos-v1'


https://www.sbert.net/docs/sentence_transformer/pretrained_models.html


In [4]:
class QdrantVectorStoreWrapper():
    EMBEDDINGS_MODEL = 'sentence-transformers/multi-qa-mpnet-base-dot-v1'
    def __init__(self, chunk_size, overlap, k, path = None):
        self.init_text_splitter(chunk_size, overlap)
        self.init_vector_store(path)
        self.k = k
        self.sequential_doc_number = 1

    def init_text_splitter(self, chunk_size, overlap):
        embedding_tokenizer = AutoTokenizer.from_pretrained(self.EMBEDDINGS_MODEL)

        self.text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
            tokenizer=embedding_tokenizer,
            chunk_size=chunk_size,
            chunk_overlap=overlap,
        )

    def init_vector_store(self, path):
      if path:
          self.client = QdrantClient(path=path)
      else:
          self.client = QdrantClient(":memory:")
      collection_name = f"rag_tech_db_{CHUNK_SIZE}"
      self.client.create_collection(
          collection_name=collection_name,
          vectors_config=VectorParams(size=768, distance=Distance.DOT)
      )
      base_embeddings = HuggingFaceEmbeddings(model_name=self.EMBEDDINGS_MODEL)
      self.vector_store = QdrantVectorStore(
          client=self.client,
          embedding=base_embeddings,
          collection_name=collection_name,
          distance=Distance.DOT
      )

    def retrieve(self, query: str, k: int = None):
        if k == None:
            k = self.k
        results = self.vector_store.similarity_search_with_score(query, k=k)
        return "\n\n".join([doc[0].page_content for doc in results])

    def add_documents(self, documents=[], metadata={}):
        splits = self.text_splitter.split_documents(documents)
        for idx, text in enumerate(splits):
            splits[idx].metadata['split_id'] = idx
            splits[idx].metadata['doc_num'] = self.sequential_doc_number
            for k, v in metadata.items():
                splits[idx].metadata[k] = v
        self.vector_store.add_documents(splits)
        print(f"Successfully added {len(splits)} splits as Document #{self.sequential_doc_number}.")
        print(f"Total vector count: {self.count()}")
        self.sequential_doc_number += 1

    def count(self):
        return self.vector_store.client.count(self.vector_store.collection_name)

    def load_arxiv(self):
      arxiv_numbers = ('2005.11401', '2104.07567', '2104.09864', '2105.03011', '2106.09685', '2203.02155', '2203.15556', '2211.09260',
                 '2211.12561', '2212.09741', '2305.14314', '2305.18290', '2306.15595', '2309.08872', '2309.15217', '2310.06825',
                 '2310.11511', '2311.08377', '2312.05708', '2401.06532', '2401.17268', '2402.01306', '2402.19473', '2406.04744',
                 '2312.10997', '2410.12812', '2410.15944', '2411.16594', '2404.00657', '2601.07711', '2507.09477', '2506.10408')


      for identifier in arxiv_numbers:
          # Construct URL using the arXiv unique identifier
          arx_url = f"https://arxiv.org/pdf/{identifier}.pdf"
          article_pages = []

          time.sleep(2)
          arx_loader = PyMuPDFLoader(arx_url)
          arx_pages = arx_loader.load()
          for page_num in range(len(arx_pages)):
              page = arx_pages[page_num]
              page.metadata['page_num'] = page_num
              page.metadata['doc_source'] = "ArXiv"
              page.metadata['id'] = identifier
              article_pages.append(page)
          self.add_documents(article_pages)
      print("📚 Successfully added all ArXiv paper")

    def load_wikipedia(self):
        wiki_queries = [
            "Generative AI",
            "Information Retrieval",
            "Large Language Models",
            "Retrieval Augmented Generation"
        ]

        for query in wiki_queries:
            wiki_docs = self.retreive_wikipedia_documents(query)
            if wiki_docs:
                self.add_documents(wiki_docs)
                print(f"Successfully added Wikipedia article on'{query}' to the vector store.")

        print("📚 Successfully added all Wikipedia entries")

    def load_web_pages(self):
      web_loader = WebBaseLoader(web_paths=(
              "https://lilianweng.github.io/posts/2023-06-23-agent/",
              "https://lilianweng.github.io/posts/2020-10-29-odqa/",
              "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
              "https://lilianweng.github.io/posts/2018-06-24-attention/",
              "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
              "https://lilianweng.github.io/posts/2024-07-07-hallucination/",
              "https://huyenchip.com/2025/01/16/ai-engineering-pitfalls.html",
              "https://blog.langchain.com/the-rise-of-context-engineering/"
          ),
          bs_kwargs=dict(
              parse_only=bs4.SoupStrainer(
                  class_=("post-content", "post-title", "post-header")
              )
          ),
      )
      web_documents = web_loader.load()
      for page in web_documents:
          self.add_documents([page], metadata={'id': page.metadata['source']})
      print('Number of Web documents: ', len(web_documents))
      print("📚 Successfully added all Web pages")

    def retreive_wikipedia_documents(self, query, retry=True):
        try:
            wiki_docs = WikipediaLoader(query=query, load_max_docs=2).load()
            for idx, text in enumerate(wiki_docs):
                wiki_docs[idx].metadata['doc_source'] = "Wikipedia"
                wiki_docs[idx].metadata['id'] = wiki_docs[idx].metadata['title']
            print('query = ', query)
            print('  Number of documents: ', len(wiki_docs))
            return wiki_docs
        except Exception as e:
            time.sleep(5)
            if retry:
              wiki_splits = self.retreive_wikipedia_documents(query, retry=False)
            else:
              wiki_splits = []

In [5]:
CHUNK_SIZE = 150
OVERLAP = 30
K=20
vector_store = QdrantVectorStoreWrapper(chunk_size=CHUNK_SIZE,
                                        overlap=OVERLAP,
                                        k=K,
                                        path="/content/qdrant_storage")
vector_store.load_wikipedia()
vector_store.load_arxiv()
vector_store.load_web_pages()

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

query =  Generative AI
  Number of documents:  2
Successfully added 11 splits as Document #1.
Total vector count: count=11
Successfully added Wikipedia article on'Generative AI' to the vector store.
query =  Information Retrieval
  Number of documents:  2
Successfully added 16 splits as Document #2.
Total vector count: count=27
Successfully added Wikipedia article on'Information Retrieval' to the vector store.


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (556 > 512). Running this sequence through the model will result in indexing errors


query =  Large Language Models
  Number of documents:  2
Successfully added 10 splits as Document #3.
Total vector count: count=37
Successfully added Wikipedia article on'Large Language Models' to the vector store.
query =  Retrieval Augmented Generation
  Number of documents:  2
Successfully added 17 splits as Document #4.
Total vector count: count=54
Successfully added Wikipedia article on'Retrieval Augmented Generation' to the vector store.
📚 Successfully added all Wikipedia entries
Successfully added 157 splits as Document #5.
Total vector count: count=211
Successfully added 199 splits as Document #6.
Total vector count: count=410
Successfully added 109 splits as Document #7.
Total vector count: count=519
Successfully added 104 splits as Document #8.
Total vector count: count=623
Successfully added 211 splits as Document #9.
Total vector count: count=834
Successfully added 408 splits as Document #10.
Total vector count: count=1242
Successfully added 233 splits as Document #11.
Tota

In [6]:
# 1. Back up your data at the end of a session
!tar -czf "/content/drive/MyDrive/Colab Data/MIDS-267-A5/qdrant_150_30.tar.gz" -C /content qdrant_storage


In [7]:
marketing_persona = {
    "name": "marketing",
    "description": """This user is a marketer who will ask questions about generative AI in order to better understand the products and the field as a whole.
                      They prefer high level answers that explain concept over technical detail.
                      You will help them find accurate, approved messaging about generative AI features, competitive positioning, and technical capabilities to accelerate content production.""",
}

research_persona = {
    "name": "research",
    "description": """This user is an engineer, who requires detailed technical information when they ask questions.
                      You will help them by writing questions about generative AI concepts, internal system architecture, and implementation details."""
}

llm_template = """Write a question that a {name} professional would ask based on the following context.
{description}

The question must:
- Be related to generative AI
- Be answerable directly from this passage
- Use your own phrasing, do not copy the context word for word.

Context:
{context}

Respond only with the question nothing else.
Question:"""
llm_prompt_template = PromptTemplate(template=llm_template, input_variables=["object"])

cohere_chat_model = ChatCohere(cohere_api_key=COHERE_API_KEY)

cohere_chain = llm_prompt_template | cohere_chat_model | StrOutputParser()


In [8]:
# given a document from the vector store
# given a persona
# create a question based on the context-persona
# save the document number of the question
# run a similarity search, check if it pulls the correct document
import random
random.seed(42)

def eval_context_persona(context, persona, persona_name):
    document_number = context.payload['metadata']['id']
    persona_context = persona.copy()
    persona_context['context'] = context.payload['page_content']
    time.sleep(0.1)
    question = cohere_chain.invoke(persona_context)
    vector_store_result = vector_store.vector_store.similarity_search_with_score(question, k=5)
    question_score = vector_store_result[0][1]
    question_id = vector_store_result[0][0].metadata['id']
    recall = [1 if x[0].metadata['id'] == document_number else 0 for x in vector_store_result]
    return {
        'persona': persona_name,
        'question_score': question_score,
        'question_id': question_id,
        'recall': recall,
        'question': question
    }


points, _ = vector_store.vector_store.client.scroll(
    collection_name=vector_store.vector_store.collection_name,
    limit=1000, with_payload=True)

if not points:
    print("Error: The vector store is empty. No documents were found in the collection.")
    print("Please ensure that the 'vector_store.load_wikipedia()', 'vector_store.load_arxiv()', and 'vector_store.load_web_pages()' methods executed successfully and added documents in cell V_BR4rgYrCGn.")
else:
    output = []
    total_questions = 0
    research_scores = []
    marketing_scores = []
    research_recall_1 = []
    research_recall_3 = []
    research_recall_5 = []
    marketing_recall_1 = []
    marketing_recall_3 = []
    marketing_recall_5 = []

    for _ in range(100):
        context = random.choice(points)

        research_result = eval_context_persona(context, research_persona, 'research')
        marketing_result = eval_context_persona(context, marketing_persona, 'marketing')
        for result in [research_result, marketing_result]:
            result['id'] = context.payload['metadata']['id']
            result['context_id'] = context.id

        output.append(research_result)
        output.append(marketing_result)

        research_scores.append(research_result['question_score'])
        marketing_scores.append(marketing_result['question_score'])

        research_recall_1.append(any(research_result['recall'][:1]))
        research_recall_3.append(any(research_result['recall'][:3]))
        research_recall_5.append(any(research_result['recall'][:5]))
        marketing_recall_1.append(any(marketing_result['recall'][:1]))
        marketing_recall_3.append(any(marketing_result['recall'][:3]))
        marketing_recall_5.append(any(marketing_result['recall'][:5]))


    print(f"Average Research Score = {np.mean(research_scores):.3f}")
    print(f"Average Marketing Score = {np.mean(marketing_scores):.3f}")
    print(f"Average Research Recall@1 = {np.mean(research_recall_1):.3f}")
    print(f"Average Research Recall@3 = {np.mean(research_recall_3):.3f}")
    print(f"Average Research Recall@5 = {np.mean(research_recall_5):.3f}")
    print(f"Average Marketing Recall@1 = {np.mean(marketing_recall_1):.3f}")
    print(f"Average Marketing Recall@3 = {np.mean(marketing_recall_3):.3f}")
    print(f"Average Marketing Recall@5 = {np.mean(marketing_recall_5):.3f}")


Average Research Score = 29.598
Average Marketing Score = 28.441
Average Research Recall@1 = 0.790
Average Research Recall@3 = 0.880
Average Research Recall@5 = 0.880
Average Marketing Recall@1 = 0.530
Average Marketing Recall@3 = 0.690
Average Marketing Recall@5 = 0.730


In [9]:
import json
path = f"/content/drive/MyDrive/Colab Data/MIDS-267-A5/vector_store_eval_chunk_{CHUNK_SIZE}_olap_{OVERLAP}_output.json"
json.dump(output, open(path, "w"), indent=2)

In [10]:
import json
import sys
from pathlib import Path

import numpy as np

PERSONAS = ["research", "marketing"]


def summarize(records, persona):
    rows = [r for r in records if r.get("persona") == persona]
    if not rows:
        return None

    scores = [r["question_score"] for r in rows]

    # recall@k: did the gold doc appear anywhere in the top k?
    def recall_at(k):
        return np.mean([any(r["recall"][:k]) for r in rows])

    # density: what fraction of the top-5 came from the gold doc?
    density = np.mean([sum(r["recall"]) / len(r["recall"]) for r in rows])

    return {
        "n": len(rows),
        "score": np.mean(scores),
        "r1": recall_at(1),
        "r3": recall_at(3),
        "r5": recall_at(5),
        "density": density,
    }


header = (
    f"{'config':<28}{'persona':<12}{'n':>5}{'score':>8}"
    f"{'R@1':>7}{'R@3':>7}{'R@5':>7}{'top5_density':>14}"
)
print(header)
print("-" * len(header))

paths = [
    "/content/drive/MyDrive/Colab Data/MIDS-267-A5/vector_store_eval_chunk_250_olap_50_output.json",
    "/content/drive/MyDrive/Colab Data/MIDS-267-A5/vector_store_eval_chunk_150_olap_30_output.json"
]

for path in paths:
    records = json.loads(Path(path).read_text())
    config = Path(path).stem

    for persona in PERSONAS:
        m = summarize(records, persona)
        if m is None:
            continue
        print(
            f"{config:<28.28}{persona:<12}{m['n']:>5}{m['score']:>8.1f}"
            f"{m['r1']:>7.2f}{m['r3']:>7.2f}{m['r5']:>7.2f}{m['density']:>14.2f}"
        )


config                      persona         n   score    R@1    R@3    R@5  top5_density
----------------------------------------------------------------------------------------
vector_store_eval_chunk_250_research      100    30.2   0.82   0.90   0.91          0.61
vector_store_eval_chunk_250_marketing     100    28.9   0.64   0.80   0.83          0.52
vector_store_eval_chunk_150_research      100    29.6   0.79   0.88   0.88          0.54
vector_store_eval_chunk_150_marketing     100    28.4   0.53   0.69   0.73          0.41


## Test of Vector Store Hyper Parameters

Tested against both personas - research and marketing.

**Option A:** 250 tokens per chunk and overlap 40, this is the max recommended with the sentence transformers model

**Option B:** 150 tokens per chunk and overlap 30, a reduced size to compare

\>250 was not tested based on the documentation. While it can handle up to 512 tokens, that was not advised and the model was train on chunks up to 250 tokens.

### Procedure
Populate the vector store with hyperparameters chosen for chunk size and overlap.

Select 100 random chunks from the vector store, these are the Original documents.

For each Original docuemnt, use Cohere to generate a question given the context and the persona. Research and marketing each generate 1 question per Original document.

Query the vector store with that Question.

Return the top 5 results analyze the Returned documents.


### Metrics

Score - semantic similarity score of a question and a chunk recalled
Recall@1, Recall@3, Recall@5 - Does the document that was used to generate the question appear in the top N documents of the query based on that question?
Top 5 Density - Out of the top 5 documents, how many come from the original?

### Findings

Research had a clear winner - 250 chunk size, 40 overlap. Across all metrics, the 250 chunk size is the winner

Marketing also slightly better with 250/40, though overall meaningfully worse than research.

| Config | Persona | n | Score | R@1 | R@3 | R@5 | Top-5 Density |
|---|---|---:|---:|---:|---:|---:|---:|
| 250 / 40 | Research | 100 | 30.0 | 0.76 | 0.86 | 0.89 | 0.49 |
| 150 / 30 | Research | 100 | 29.8 | 0.71 | 0.82 | 0.86 | 0.41 |
| 250 / 40 | Marketing | 100 | 28.6 | 0.56 | 0.67 | 0.69 | 0.35 |
| 150 / 30 | Marketing | 100 | 28.6 | 0.55 | 0.66 | 0.70 | 0.32 |

### Conclusion
Given that research had a clear winner (250 chunk size / 40 overlap), and marketing was closer to neutral, I will proceed with 250/40 for the vector store hyperparameters in the RAG system.



